Current v3 summary: compare delivery calendar dates; same promised day is on time. PostgreSQL ingestion is unchanged. Full approval-stage eligibility is applied in Task2/corrected_data.py.


# MLOps Training 2026/2027 — Task 1

## Get the Data Into a Database

This notebook loads the Olist Brazilian E-Commerce CSV files into a local PostgreSQL database and verifies the tables, schema, relationships, and delivery problem. Full EDA, feature engineering, and model training belong to later tasks.

Official dataset page: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


## Step 1 — Read and understand

The dataset is relational, not one ready-made ML table. Different files represent orders, customers, order items, payments, reviews, products, sellers, geolocation, and category translations. The principal keys are `order_id`, `customer_id`, `product_id`, and `seller_id`.

A future ML table must have one row per order. One-to-many tables must be aggregated by `order_id` before joining, otherwise numeric values can be duplicated. Reviews and actual delivery dates are useful for analysis but may cause leakage if used as features before delivery.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "Task1").is_dir():
    project_root = project_root.parent

src_dir = project_root / "Task1" / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config import load_settings

settings = load_settings()
archive_path = settings.data_dir
if not archive_path.exists():
    raise FileNotFoundError(f"Dataset folder not found: {archive_path}")

print(f"pandas version: {pd.__version__}")
print(f"Dataset folder: {archive_path}")


In [ ]:
csv_files = sorted(archive_path.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV files found")
file_manifest = pd.DataFrame({"file_name": [p.name for p in csv_files], "size_mb": [round(p.stat().st_size / 1024**2, 2) for p in csv_files]})
print(f"Found {len(csv_files)} CSV files")
display(file_manifest)


## Step 2 — Connect to PostgreSQL

Connection settings are read from environment variables. The password is requested interactively when `OLIST_DB_PASSWORD` is not set, so credentials are not stored in the notebook.


In [ ]:
import getpass
from sqlalchemy import create_engine, text
from config import build_database_url

DB_PASSWORD = os.getenv("OLIST_DB_PASSWORD") or getpass.getpass("PostgreSQL password: " )
engine = create_engine(build_database_url(settings, DB_PASSWORD), pool_pre_ping=True)

with engine.connect() as connection:
    version = connection.execute(text("SELECT version();")).scalar_one()
print("PostgreSQL connection successful.")
print(version)


## Step 2 — Load the CSV files

Each CSV is loaded into a table. The `_dataset.csv` suffix is removed from dataset filenames. Replacing tables is intentional for a reproducible local rebuild. Constraints can be applied afterward with `Relationships.sql`.


In [ ]:
def table_name_from_file(file_name: str) -> str:
    suffix = "_dataset.csv" if file_name.endswith("_dataset.csv") else ".csv"
    return file_name[:-len(suffix)]

load_rows = []
for csv_path in csv_files:
    table_name = table_name_from_file(csv_path.name)
    df = pd.read_csv(csv_path)
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10000)
    load_rows.append({"file_name": csv_path.name, "table_name": table_name, "rows_loaded": len(df), "columns": len(df.columns)})
load_manifest = pd.DataFrame(load_rows).sort_values("table_name").reset_index(drop=True)
display(load_manifest)
print("All CSV files loaded successfully.")


## Step 3 — Verify the database and schema


In [ ]:
from sqlalchemy import inspect

expected_tables = {table_name_from_file(p.name) for p in csv_files}
inspector = inspect(engine)
actual_tables = set(inspector.get_table_names())
missing = expected_tables - actual_tables
if missing:
    raise AssertionError(f"Missing tables: {sorted(missing)}")

row_counts = []
schema_rows = []
with engine.connect() as connection:
    for table_name in sorted(expected_tables):
        count = connection.execute(text(f'SELECT COUNT(*) FROM "{table_name}"')).scalar_one()
        row_counts.append({"table_name": table_name, "row_count": count})
        for column in inspector.get_columns(table_name):
            schema_rows.append({"table_name": table_name, "column_name": column["name"], "data_type": str(column["type"])})
row_counts = pd.DataFrame(row_counts)
schema_df = pd.DataFrame(schema_rows)
display(row_counts)
print(f"Verified {len(row_counts)} tables and {row_counts.row_count.sum():,} total rows.")
display(schema_df)


## Step 3.1 — Test the key relationships

Each query tests one relationship independently. The output is limited only for readability.


In [ ]:
join_queries = {
    "orders_customers": "SELECT o.order_id, o.customer_id, o.order_status, c.customer_city, c.customer_state FROM olist_orders o JOIN olist_customers c ON o.customer_id = c.customer_id LIMIT 5;",
    "orders_items_products": "SELECT o.order_id, oi.product_id, p.product_category_name, oi.price, oi.freight_value FROM olist_orders o JOIN olist_order_items oi ON o.order_id = oi.order_id JOIN olist_products p ON oi.product_id = p.product_id LIMIT 5;",
    "items_sellers": "SELECT oi.order_id, oi.order_item_id, oi.seller_id, s.seller_city, s.seller_state FROM olist_order_items oi JOIN olist_sellers s ON oi.seller_id = s.seller_id LIMIT 5;",
    "orders_payments": "SELECT o.order_id, op.payment_type, op.payment_value, op.payment_installments FROM olist_orders o JOIN olist_order_payments op ON o.order_id = op.order_id LIMIT 5;",
}
for name, query in join_queries.items():
    print(f"\n--- {name} ---")
    display(pd.read_sql_query(query, engine))


## Step 3.2 — Understand the delivery problem

For delivered orders with valid dates, `is_late` is 1 when the actual delivery date is after the estimated date, and 0 when it is on or before the estimate. The target is defined here for analysis; future features must represent information available at prediction time.


In [ ]:
status_query = "SELECT order_status, COUNT(*) AS order_count, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage FROM olist_orders GROUP BY order_status ORDER BY order_count DESC;"
status_df = pd.read_sql_query(status_query, engine)
display(status_df)

late_query = "SELECT COUNT(*) AS delivered_orders, COUNT(*) FILTER (WHERE CAST(order_delivered_customer_date AS timestamp)::date > CAST(order_estimated_delivery_date AS timestamp)::date) AS late_orders, COUNT(*) FILTER (WHERE CAST(order_delivered_customer_date AS timestamp)::date <= CAST(order_estimated_delivery_date AS timestamp)::date) AS on_time_orders, ROUND(100.0 * COUNT(*) FILTER (WHERE CAST(order_delivered_customer_date AS timestamp)::date > CAST(order_estimated_delivery_date AS timestamp)::date) / COUNT(*), 2) AS late_percentage FROM olist_orders WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL AND order_estimated_delivery_date IS NOT NULL;"
late_summary = pd.read_sql_query(late_query, engine)
display(late_summary)


## Optional database constraints

After ingestion and verification, `Relationships.sql` may be run to add primary and foreign keys. The category translation table is intentionally not linked by a foreign key because it does not cover every product category.


## Final completion check

Task 1 is complete when the database runs locally with data inside, the tables can be queried and joined, the relationships are understood, and the late-delivery problem is defined.


In [ ]:
required = {"database_connection": False, "all_tables_present": False, "join_test_passed": False, "prediction_problem_defined": False}
with engine.connect() as connection:
    connection.execute(text("SELECT 1"))
required["database_connection"] = True
required["all_tables_present"] = expected_tables.issubset(set(inspect(engine).get_table_names()))
join_count = pd.read_sql_query("SELECT COUNT(*) AS joined_rows FROM olist_orders o JOIN olist_customers c ON o.customer_id = c.customer_id;", engine).iloc[0]["joined_rows"]
required["join_test_passed"] = join_count > 0
order_columns = set(schema_df.loc[schema_df.table_name == "olist_orders", "column_name"])
required["prediction_problem_defined"] = {"order_delivered_customer_date", "order_estimated_delivery_date"}.issubset(order_columns)
check_df = pd.DataFrame({"criterion": list(required), "passed": list(required.values())})
display(check_df)
assert all(required.values()), f"Completion check failed: {required}"
print("ALL TASK 1 COMPLETION CRITERIA PASSED")
